In [2]:
import pandas as pd

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount', 'total_amount']
df = pd.read_parquet(url, columns=columns).head(1000)
test = pd.read_parquet(url, columns=columns)
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [3]:
print(len(test) * 0.01)

494.16


In [4]:
print(len(test) * 3)

148248


In [2]:
print(df.iloc[0])


lpep_pickup_datetime     2025-10-01 00:21:47
lpep_dropoff_datetime    2025-10-01 00:24:37
PULocationID                             247
DOLocationID                              69
passenger_count                          1.0
trip_distance                            0.7
tip_amount                               1.7
total_amount                            10.0
Name: 0, dtype: object


In [5]:
from dataclasses import dataclass

@dataclass
class Ride:
    lpep_pickup_datetime: int  # epoch milliseconds
    lpep_dropoff_datetime: int
    PULocationID: int
    DOLocationID: int
    passenger_count: float
    trip_distance: float
    tip_amount: float
    total_amount: float

In [9]:
def ride_from_row(row):
    return Ride(
        lpep_pickup_datetime=int(row['lpep_pickup_datetime'].timestamp() * 1000),
        lpep_dropoff_datetime=int(row['lpep_pickup_datetime'].timestamp() * 1000),
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        passenger_count=float(row['passenger_count']),
        trip_distance=float(row['trip_distance']),
        tip_amount=float(row['tip_amount']),
        total_amount=float(row['total_amount']),
    )

In [10]:
ride = ride_from_row(df.iloc[0])
ride

Ride(lpep_pickup_datetime=1759278107000, lpep_dropoff_datetime=1759278107000, PULocationID=247, DOLocationID=69, passenger_count=1.0, trip_distance=0.7, tip_amount=1.7, total_amount=10.0)

In [14]:
import json
from kafka import KafkaProducer

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

In [15]:
import dataclasses

topic_name = 'green-trips'

producer.send(topic_name, value=dataclasses.asdict(ride))
producer.flush()

In [16]:
def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')

In [19]:
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [20]:
producer.send(topic_name, value=ride)
producer.flush()

In [21]:
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

Sent: Ride(lpep_pickup_datetime=1759278107000, lpep_dropoff_datetime=1759278107000, PULocationID=247, DOLocationID=69, passenger_count=1.0, trip_distance=0.7, tip_amount=1.7, total_amount=10.0)
Sent: Ride(lpep_pickup_datetime=1759277643000, lpep_dropoff_datetime=1759277643000, PULocationID=66, DOLocationID=25, passenger_count=1.0, trip_distance=1.61, tip_amount=2.78, total_amount=16.68)
Sent: Ride(lpep_pickup_datetime=1759277804000, lpep_dropoff_datetime=1759277804000, PULocationID=244, DOLocationID=244, passenger_count=1.0, trip_distance=0.0, tip_amount=2.2, total_amount=13.2)
Sent: Ride(lpep_pickup_datetime=1759277256000, lpep_dropoff_datetime=1759277256000, PULocationID=95, DOLocationID=170, passenger_count=1.0, trip_distance=10.37, tip_amount=11.31, total_amount=67.85)
Sent: Ride(lpep_pickup_datetime=1759266629000, lpep_dropoff_datetime=1759266629000, PULocationID=82, DOLocationID=138, passenger_count=1.0, trip_distance=4.07, tip_amount=6.82, total_amount=34.12)
Sent: Ride(lpep_pic